# Model Verification

This notebook collects some experiments used to verify the validity of the default [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) model.

> **Note:** The model is still under development, so the values shown here might still change.
> Also, there are a couple of options that can modify the default [xvamp.model.Duan2010](../xvamp/model.rst#xvamp.model.Duan2010) behavior, which are documented in the API, but not shown here.

## Venus Brightness Temperature

We start by initializing our default model:

In [ ]:
# some basic imports
import numpy as np
import matplotlib.pyplot as plt
from cmcrameri import cm
from astropy.units import Quantity
from astropy.visualization import quantity_support

# import reference model
from xvamp.model import Duan2010

# instantiate
model = Duan2010()

# initialize pretty plotting
quantity_support()
%config InlineBackend.figure_formats = ["svg", "pdf"]

The brightness temperature calculation requires the knowledge of the brightness
temperature at the surface, which is not something our model can compute.
However, we know the temperature of the atmosphere at the surface, and the surface
should be somewhat equilibrated with that temperature. So, we simply test our
model using a range of surface brightness temperatures that are derived from the
atmosphere's temperature:

In [ ]:
# import brightness computation function
from xvamp.utils import get_brightness_temperature

# define range of factors to apply to atmosphere temperature
# to derive surface brightness temperature
factor_range = np.linspace(0.5, 1.0, num=51)

# select some altitude levels for which to compute everything
ix_alt = np.arange(1, 15, 2)

# define range of surface brightness temperatures to assume
T_B_surface = factor_range[:, None] * model.temperature[ix_alt][None, :]

# for each assumed element in T_B_surface, get the total brightness temperature
T_B = Quantity(
    [
        get_brightness_temperature(
            model.altitude[i:],
            model.temperature[i:],
            model.refraction[i:],
            model.absorption[i:],
            Quantity(30, "°"),
            T_B_surface[:, ii],
        )
        for ii, i in enumerate(ix_alt)
    ]
)

In [ ]:
# import plotting functionality
from matplotlib.colors import Normalize

# start with the Ho & Kaufman ranges for X-band
plt.axhspan(Quantity(578, "K"), Quantity(657, "K"), fc="k", alpha=0.1, ec=None)
plt.axhspan(Quantity(500, "K"), Quantity(660, "K"), fc="k", alpha=0.1, ec=None)
# add our derived curves, colored by terrain altitude
norm = Normalize(
    vmin=model.altitude[ix_alt[0]].value, vmax=model.altitude[ix_alt[-1]].value
)
for ii, i in enumerate(ix_alt):
    plt.plot(
        factor_range,
        T_B[ii, :],
        c=cm.batlow(norm(model.altitude[i].value)),
        lw=2,
        label=f"{model.altitude[i]:.0f}",
    )
# make pretty
plt.xlabel("Surface Brightness Temperature Factor [-]")
plt.ylabel("Venus Brightness Temperature [K]")
plt.legend(ncol=2)
plt.xlim(factor_range[0], factor_range[-1])
plt.ylim(400, 800)